# Topic Extraction EDA

This notebook is a raw-data audit for `nyc311` topic extraction.

Setup for local use:
- run `uv sync --extra science --group notebooks`
- select the project `.venv` / `nyc311` kernel in Jupyter or VS Code

It shows how to:
- inspect the complaint-type distribution from a live 311 slice
- measure built-in topic-rule coverage
- inspect unmatched descriptors so topic logic does not silently lose signal
- add custom rules for a new complaint type
- render anomaly and report-style summaries directly inside the notebook

In [ ]:
from datetime import date

from IPython.display import Markdown, display

import nyc311

records = nyc311.fetch_service_requests(
    filters=nyc311.ServiceRequestFilter(
        start_date=date(2025, 1, 1),
        end_date=date(2025, 3, 31),
        geography=nyc311.GeographyFilter("borough", nyc311.BOROUGH_BROOKLYN),
    ),
    socrata_config=nyc311.SocrataConfig(page_size=1000, max_pages=5),
)

records_df = nyc311.records_to_dataframe(records)
records_df.head()

In [ ]:
complaint_distribution = (
    records_df["complaint_type"]
    .value_counts()
    .rename_axis("complaint_type")
    .reset_index(name="count")
)

display(complaint_distribution.head(15))

plot_df = complaint_distribution.head(15).sort_values("count")
try:
    import matplotlib.pyplot as plt

    ax = plot_df.plot.barh(
        x="complaint_type",
        y="count",
        figsize=(9, 5),
        color="#4C78A8",
        legend=False,
        title="Top complaint types in the sample",
    )
    ax.set_xlabel("Count")
    ax.set_ylabel("Complaint type")
    plt.show()
except ImportError:
    display(plot_df)

In [ ]:
coverage_rows = []
for complaint_type in nyc311.supported_topic_queries():
    coverage = nyc311.analyze_topic_coverage(
        records,
        nyc311.TopicQuery(complaint_type=complaint_type, top_n=10),
    )
    coverage_rows.append(coverage)

coverage_df = nyc311.coverage_to_dataframe(coverage_rows).sort_values(
    ["coverage_rate", "total_records"],
    ascending=[False, False],
)
display(coverage_df)

plot_df = coverage_df.sort_values("coverage_rate")
try:
    import matplotlib.pyplot as plt

    ax = plot_df.plot.barh(
        x="complaint_type",
        y="coverage_rate",
        figsize=(9, 4),
        color="#59A14F",
        legend=False,
        title="Coverage rate by built-in topic ruleset",
    )
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Coverage rate")
    ax.set_ylabel("Complaint type")
    plt.show()
except ImportError:
    display(plot_df[["complaint_type", "coverage_rate", "other_records"]])

## Auditing Rule Coverage

The built-in topic constants are intended to be data-driven starting points, not hidden magic.

If a ruleset falls back to `other`, inspect the unmatched descriptors directly so you can decide whether to:
- keep the descriptor in `other`
- add a better built-in rule
- supply custom rules in your own workflow

In [ ]:
illegal_parking_coverage = nyc311.analyze_topic_coverage(
    records,
    nyc311.TopicQuery("Illegal Parking", top_n=10),
)

unmatched_df = nyc311.records_to_dataframe(
    [
        nyc311.ServiceRequestRecord(
            service_request_id=f"unmatched-{index}",
            created_date=date(2025, 1, 1),
            complaint_type="Illegal Parking",
            descriptor=descriptor,
            borough=nyc311.BOROUGH_BROOKLYN,
            community_district="01 BROOKLYN",
        )
        for index, (descriptor, _count) in enumerate(
            illegal_parking_coverage.top_unmatched_descriptors,
            start=1,
        )
    ]
)[["descriptor"]]
unmatched_df["count"] = [
    count for _descriptor, count in illegal_parking_coverage.top_unmatched_descriptors
]
display(unmatched_df)

In [ ]:
custom_rules = (
    ("hydrant_issue", ("hydrant", "low water pressure")),
    ("leak", ("leak", "leaking")),
)

synthetic_records = [
    nyc311.ServiceRequestRecord(
        service_request_id="demo-1",
        created_date=date(2025, 1, 1),
        complaint_type="Water System",
        descriptor="Low water pressure near hydrant",
        borough=nyc311.BOROUGH_BROOKLYN,
        community_district="01 BROOKLYN",
    ),
    nyc311.ServiceRequestRecord(
        service_request_id="demo-2",
        created_date=date(2025, 1, 2),
        complaint_type="Water System",
        descriptor="Leaking hydrant on corner",
        borough=nyc311.BOROUGH_BROOKLYN,
        community_district="01 BROOKLYN",
    ),
    nyc311.ServiceRequestRecord(
        service_request_id="demo-3",
        created_date=date(2025, 1, 3),
        complaint_type="Water System",
        descriptor="Pressure issue in building basement",
        borough=nyc311.BOROUGH_BROOKLYN,
        community_district="01 BROOKLYN",
    ),
]

before_custom_rules = nyc311.analyze_topic_coverage(
    synthetic_records,
    nyc311.TopicQuery("Water System", top_n=10),
)
after_custom_rules = nyc311.analyze_topic_coverage(
    synthetic_records,
    nyc311.TopicQuery("Water System", top_n=10),
    custom_rules=custom_rules,
)

custom_coverage_df = nyc311.coverage_to_dataframe([before_custom_rules, after_custom_rules])
custom_coverage_df.index = ["Fallback grouping", "Custom rules"]
display(custom_coverage_df)

In [ ]:
noise_assignments = nyc311.extract_topics(
    records,
    nyc311.TopicQuery("Noise - Residential", top_n=10),
)
noise_summaries = nyc311.aggregate_by_geography(
    noise_assignments,
    geography="community_district",
)
noise_summaries_df = nyc311.summaries_to_dataframe(noise_summaries)
display(noise_summaries_df.head(10))

dominant_noise_df = noise_summaries_df[noise_summaries_df["is_dominant_topic"]].sort_values(
    "share_of_geography",
    ascending=False,
)
try:
    import matplotlib.pyplot as plt

    ax = dominant_noise_df.head(12).sort_values("share_of_geography").plot.barh(
        x="geography_value",
        y="share_of_geography",
        figsize=(9, 5),
        color="#F28E2B",
        legend=False,
        title="Dominant noise-topic share by district",
    )
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Share of geography")
    ax.set_ylabel("Community district")
    plt.show()
except ImportError:
    display(dominant_noise_df[["geography_value", "topic", "share_of_geography"]].head(12))

In [ ]:
anomalies = nyc311.detect_anomalies(
    noise_summaries,
    nyc311.AnalysisWindow(days=30),
    z_threshold=1.5,
)
anomalies_df = nyc311.anomalies_to_dataframe(anomalies)

resolution_gaps = nyc311.analyze_resolution_gaps(
    records,
    [record for record in records if record.resolution_description is not None],
)
resolution_df = nyc311.gaps_to_dataframe(resolution_gaps)

display(Markdown(
    "\n".join(
        [
            "## Inline Audit Summary",
            f"- Complaint types audited: {coverage_df.shape[0]}",
            f"- Best built-in coverage rate: {coverage_df['coverage_rate'].max():.1%}",
            f"- Noise anomaly flags above threshold: {int(anomalies_df['is_anomaly'].sum())}",
            f"- Resolution rows available: {resolution_df.shape[0]}",
        ]
    )
))

display(anomalies_df.head(10))

plot_df = anomalies_df.head(10).sort_values("z_score")
try:
    import matplotlib.pyplot as plt

    ax = plot_df.plot.barh(
        x="geography_value",
        y="z_score",
        figsize=(9, 4),
        color="#E15759",
        legend=False,
        title="Top anomaly scores in the noise topic audit",
    )
    ax.set_xlabel("z-score")
    ax.set_ylabel("Community district")
    plt.show()
except ImportError:
    display(plot_df[["geography_value", "topic", "z_score", "is_anomaly"]])